# Module 4: LangSmith — Prompt Engineering, Observability & Evaluations

> Part of the **Modular Workshops** series. Standalone, ~30 min.

We cover four parts plus a closing loop:

1. **Prompt engineering** — author, test, and version prompts in the Playground and Prompt Hub, then pull them into code with the SDK.
2. **Tracing** — generate traces with the agent from Module 2, then query them with `list_runs` + filters.
3. **Offline evaluations** — build a dataset, score the agent end-to-end (final-response with LLM-as-judge) and step-by-step (trajectory).
4. **Online evaluations** — score every new trace as it lands. Programmatic version + UI workflow.

Then **annotation queues** close the loop: route runs flagged by eval scores to a human for review.

<img src="../images/evals-conceptual.png" style="width: auto; max-height: 400px; border-radius: 8px;">

## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.models import model
from utils.langsmith_rules import (
    get_or_create_annotation_queue,
    create_run_rule,
    delete_run_rule,
)

import os, time
from datetime import datetime, timedelta, timezone
from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage, HumanMessage
from langsmith import Client, uuid7

client = Client()
print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING", "not set"))
print("Project:", os.environ.get("LANGSMITH_PROJECT", "default"))


## Part 1. Prompt Engineering — Playground & Prompt Hub

Before you can observe or evaluate an agent, you need a prompt worth shipping. LangSmith treats prompts as **versioned artifacts** — author and test them in the **Playground**, version and share them in the **Prompt Hub**, then pull them into code with the SDK. Three surfaces, one source of truth.

- **Playground** (UI) — an interactive editor: compose messages, wire up input variables, pick a model, and run.
- **Prompt Hub** (UI) — every saved prompt with full commit history, tags, and a public hub of community prompts to fork.
- **SDK** — `push_prompt` / `pull_prompt` to move prompts between code and the hub.

### 1.1 The Prompt Playground

Open **Prompts** in the LangSmith sidebar and click **+ Prompt** to land in the Playground. The left panel is your prompt — an ordered list of messages, each with a role:

- **System** — the instruction manual: persona and ground rules.
- **Human** — the user's turn.
- **AI** — a model turn, handy for few-shot examples.
- **Tool** — tool output, for testing how the model reacts to it.

Add an input variable by typing `{variable_name}` into any message (or highlight text and click **Convert to variable**). Fill in sample values in the right panel's **Inputs** box, then click **Start** to run and see the response.

**Template format.** Variables default to Python **f-string** syntax (`{topic}`). Switch to **mustache** (`{{topic}}`) from the format dropdown when you need loops, conditionals, or nested data (`{{user.name}}`) — f-strings only do flat substitution.

**Model configuration.** Click the **gear icon** next to the model name to set provider, model, temperature, and max tokens. Hit **Save As** to name a configuration — it's shared across your workspace and reusable in other LangSmith features.

**Tools.** Click **+ Tool** to attach tools: built-in ones (web search, code interpreter) or custom tools you define with a name, description, and argument schema. When the model calls a tool, the Playground shows the tool name and arguments so you can verify the call.

🔗 **Try it:** [Open Prompts in LangSmith →](https://smith.langchain.com/prompts) — then click **+ Prompt** (top right) to open the Playground.

### 1.2 Prompt Hub — save, version, share

Click **Save** in the Playground and your prompt lands in the **Prompts** table. Each prompt gets its own detail page with a two-pane layout: commit history and environments on the left, the selected commit on the right.

- **Commits** — every save is a new commit, and the full history is preserved. Toggle **Diff** (top-right) to compare a commit with its predecessor.
- **Tags** — mark a commit with a stable name (e.g. `prod`) so code can reference it without pinning a hash. Move or delete tags as the prompt evolves.
- **Environments** — reserved **Staging** and **Production** environments track which commit is live; **Promote** a commit to move it forward, or roll back from history.
- **Public hub** — search community prompts by name, use case, or model, and **fork** any of them into your workspace.

🔗 **Open in LangSmith:** [Your prompts →](https://smith.langchain.com/prompts) · [Public LangChain Hub →](https://smith.langchain.com/hub)

### 1.3 Manage prompts programmatically

Anything you do in the UI you can do from the SDK: `push_prompt` sends a prompt to the hub, and `pull_prompt` fetches it back. We'll do it in three quick steps — **push** a client-research prompt, **pull it and run it as an agent** (with `create_agent`, not a raw chain), then **version** it.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Step 1 — author the assistant's system prompt and push it to the hub.
prompt_name = "client-research-assistant"
prompt = ChatPromptTemplate([
    ("system",
     "You are a client research assistant for a wealth management firm. "
     "Look up the client, then produce a concise pre-meeting briefing: "
     "(1) a one-line summary, (2) likely priorities and risk tolerance, "
     "(3) three tailored talking points, and (4) two questions the advisor should ask. "
     "Be factual and neutral; do not give specific investment advice."),
])

url = client.push_prompt(prompt_name, object=prompt)
print("Prompt page (click to open):", url)

**Pull it back and run it — as an agent, not a chain.** `pull_prompt` returns the prompt we just pushed; we use it as the system prompt for `create_agent`, running on the workshop's shared `model`. A small mock `lookup_client` tool lets the agent fetch the profile, so the full agent loop (model → tool → model) shows up in the trace.

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def lookup_client(name: str) -> str:
    """Look up a wealth-management client's profile by name."""
    # Mock CRM lookup — swap for your real client data source in production.
    directory = {
        "Rohan Mehta": (
            "Rohan Mehta, 52, senior law-firm partner. ~$4.2M investable assets, "
            "moderately conservative. Two kids entering college within 3 years. "
            "Recently asked about tax-efficient withdrawals and ESG options."
        ),
    }
    return directory.get(name, f"No profile found for {name!r}.")


# Step 2 — pull the prompt back and run it with create_agent (uses the imported `model`).
pulled = client.pull_prompt(prompt_name)
system_prompt = pulled.format_messages()[0].content

agent = create_agent(model=model, tools=[lookup_client], system_prompt=system_prompt)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Prep me for my meeting with Rohan Mehta."}]}
)
print(result["messages"][-1].content)

**Version it.** Re-push under the same name and LangSmith records a new commit — the earlier version stays in the history.

In [ ]:
# Step 3 — re-push a tweaked version. Same name -> a new commit (full history preserved).
prompt_v2 = ChatPromptTemplate([
    ("system",
     "You are a client research assistant for a wealth management firm. "
     "Look up the client, then produce a concise pre-meeting briefing: "
     "(1) a one-line summary, (2) likely priorities and risk tolerance, "
     "(3) three tailored talking points, (4) two questions the advisor should ask, "
     "and (5) one risk or compliance flag to keep in mind. "
     "Be factual and neutral; do not give specific investment advice."),
])
url_v2 = client.push_prompt(prompt_name, object=prompt_v2)
print("New commit (click to open):", url_v2)

# Pull a specific commit with client.pull_prompt("client-research-assistant:<commit-hash>"),
# and tear down with client.delete_prompt("client-research-assistant") when you're done.

## Warm-up: Generate a few traces

Before we look at tracing and querying, let's actually produce some traces. 
We invoke the research agent from Module 2 (`agents/research_agent.py`) three times with **deliberately light** prompts — 
each one says "search at most once" so the runs finish in a few seconds instead of a few minutes.

On the trial run while building this module the warm-up took **~9 seconds total (3.1s avg per call)**. Expect similar.


In [ ]:
from agents.research_agent import build_research_agent

agent = build_research_agent()

warmup_prompts = [
    "In one sentence, what is LangGraph for? Search at most once.",
    "In one sentence, what does the LangChain v1 release add? Search at most once.",
    "In one sentence, what is Tavily? Search at most once.",
]

total = 0.0
for q in warmup_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].content[:120]}")

print(f"\nTotal: {total:.1f}s ({total/len(warmup_prompts):.1f}s avg)")


## Part 2. Tracing + Querying Traces

Set `LANGSMITH_TRACING=true` and every LLM call, tool call, and state transition lands in your tracing project — no code changes required. 
(The warm-up above already produced traces; this section pulls them back out.)

We use `client.list_runs(...)` to query them.

In [ ]:
project_name = os.environ.get("LANGSMITH_PROJECT", "modular-workshops")
try:
    project = client.read_project(project_name=project_name)
    print(f"Project: {project.name}")
    print(f"View traces: {project.url}")
except Exception as e:
    print(f"Could not read project (this is fine if first run): {e}")


### 2.1 Pull recent traces

Useful filters on `client.list_runs(...)`:

- `project_name=` — scope to one project
- `start_time=` / `end_time=` — time window
- `run_type=` — `"llm"`, `"tool"`, `"chain"`, `"retriever"`
- `error=True` — only failed runs
- `is_root=True` — only top-level traces (not their children)
- `filter=` — LangSmith filter DSL (latency, feedback, attributes...)

In [ ]:
from datetime import datetime, timedelta, timezone

# Pull the last hour of root traces from this workshop's project
since = datetime.now(timezone.utc) - timedelta(hours=1)

recent_runs = list(client.list_runs(
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    start_time=since,
    is_root=True,
    limit=20,
))

print(f"Found {len(recent_runs)} root run(s) in the last hour\n")
for r in recent_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds() if r.end_time else None
    print(f"- {r.id}  {r.name:25s}  latency={latency}s  error={r.error is not None}")


### 2.2 Filter DSL — find slow or errored runs

The `filter` argument is a small expression language. Common patterns:

- `gt(latency, 5)` — slower than 5 seconds
- `eq(status, "error")` — failed runs
- `and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))` — low-scored runs on a feedback key
- Combine with `and(...)` / `or(...)`

In [ ]:
# Find slow root runs in the last hour (>5s latency)
slow_runs = list(client.list_runs(
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    start_time=since,
    is_root=True,
    filter='gt(latency, 5)',
    limit=20,
))

print(f"{len(slow_runs)} slow root run(s) (>5s) in the last hour")
for r in slow_runs[:5]:
    latency = (r.end_time - r.start_time).total_seconds()
    print(f"  {r.name:25s}  {latency:.1f}s  {r.id}")


## Part 3. Offline Evaluations

**Offline evals** are the experiments you run on demand against a fixed dataset. 
Build a dataset once, score your agent against it whenever you change a prompt, a model, or a tool — get a clean before/after comparison.

Three pieces:
1. **Dataset** — labeled `inputs` + expected `outputs`
2. **Target function** — runs your agent on each example
3. **Evaluators** — score the output (LLM-as-judge or code-based)

### 3.1 Dataset

Same input set, two reference shapes — one for final-response judging, one for trajectory matching.

In [ ]:
examples = [
    {
        "inputs": {"query": "Write a haiku about Python to /haiku.txt"},
        "outputs": {
            "reference_answer": "A haiku about Python should be saved to /haiku.txt",
            "trajectory": ["write_file"],
        },
    },
    {
        "inputs": {"query": "Research the latest AI agent frameworks and write a brief report to /report.md"},
        "outputs": {
            "reference_answer": "A short report on recent AI agent frameworks, written to /report.md",
            "trajectory": ["task", "write_file"],
        },
    },
    {
        "inputs": {"query": "Write a short memo to /memo.md, then read it back to confirm"},
        "outputs": {
            "reference_answer": "A short memo written to /memo.md and read back",
            "trajectory": ["write_file", "read_file"],
        },
    },
    {
        "inputs": {"query": "Plan out a small research project on LangSmith, then research it and write a report to /report.md"},
        "outputs": {
            "reference_answer": "A planned-out research report on LangSmith written to /report.md",
            "trajectory": ["write_todos", "task", "write_file"],
        },
    },
]

dataset_name = "modular-workshops-evals"

if client.has_dataset(dataset_name=dataset_name):
    existing = client.read_dataset(dataset_name=dataset_name)
    client.delete_dataset(dataset_id=existing.id)
    print(f"Deleted existing dataset '{dataset_name}'")

dataset = client.create_dataset(dataset_name)
client.create_examples(
    inputs=[e["inputs"] for e in examples],
    outputs=[e["outputs"] for e in examples],
    dataset_id=dataset.id,
)
print(f"Created dataset '{dataset_name}' with {len(examples)} examples")
print(f"View: {dataset.url}")


### 3.2 Final-response eval (LLM-as-judge)

<img src="../images/final-response.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Treat the agent as a black box: did the final response satisfy the request? 
We'll build the **LLM-as-judge from scratch** so the moving parts are clear:

1. A Pydantic / TypedDict schema for the judge's output (`score`, `reasoning`)
2. A judge prompt that explains the grading criteria
3. `model.with_structured_output(...)` to force the LLM into the schema
4. An evaluator function that calls the judge and returns the score in the shape `client.evaluate` expects

In [ ]:
def run_agent_final(inputs: dict) -> dict:
    """Target function: run the agent and return its final response.

    We also concatenate any files the agent wrote into the response string so
    the judge can see actual content -- not just the agent's "saved!" message.
    """
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )

    response = result["messages"][-1].content

    file_dump = []
    for path, file_data in (result.get("files") or {}).items():
        content = file_data
        if isinstance(file_data, dict) and "content" in file_data:
            content = file_data["content"]
        if isinstance(content, list):
            content = "\n".join(content)
        file_dump.append(f"--- {path} ---\n{content}")
    if file_dump:
        response += "\n\nFiles written:\n" + "\n\n".join(file_dump)

    return {"response": response}


In [ ]:
from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage, HumanMessage

# Define the judge's output schema -- with_structured_output enforces this shape on the LLM response.
class CorrectnessGrade(TypedDict):
    """Score whether the agent's response satisfied the user's request."""
    score: bool   # True if correct/helpful, False otherwise
    reasoning: str  # one-sentence explanation

# The dataset's `reference_answer` is a SUCCESS RUBRIC, not an expected response text.
# Make that explicit to the judge so it doesn't downscore valid agent responses that
# happen to be worded differently.
correctness_judge_prompt = """You are an expert grader evaluating an AI assistant's response.

You'll see the user's request, the assistant's final response, and a RUBRIC describing what success looks like.

The rubric is NOT the expected response text -- it's the success criteria. Mark `score=True` if the assistant's response demonstrates that the criteria were met (e.g., it confirms the file was written, or shows the content that was saved). The assistant's wording doesn't need to match the rubric -- what matters is whether the underlying task was accomplished.

Mark `score=False` only if the response clearly missed the task, contained errors, or refused without good reason.
Give one short sentence of reasoning either way.
"""

# Bind the schema once -- `judge` is now a structured-output LLM.
judge = model.with_structured_output(CorrectnessGrade)

def correctness_evaluator(inputs, outputs, reference_outputs):
    grade = judge.invoke([
        SystemMessage(content=correctness_judge_prompt),
        HumanMessage(content=(
            f"User request: {inputs['query']}\n\n"
            f"Assistant response: {outputs['response']}\n\n"
            f"Success rubric: {reference_outputs['reference_answer']}"
        )),
    ])
    return {"key": "correctness", "score": int(grade["score"]), "comment": grade["reasoning"]}


In [ ]:
results = client.evaluate(
    run_agent_final,
    data=dataset_name,
    evaluators=[correctness_evaluator],
    experiment_prefix="final-response",
    max_concurrency=2,
)
print(f"View at: {results.experiment_name}")


### 3.3 Trajectory eval

<img src="../images/trajectory.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Score the **sequence of tool calls** the agent took, not just the final answer. Three evaluators:

- **`exact_match`** — did it take exactly the right steps in order?
- **`extra_steps`** — how many extra tool calls did it make?
- **`missing_steps`** — how many expected steps did it skip?

`extra_steps` and `missing_steps` use `collections.Counter` for multiset diffs — order doesn't matter for those two, but `exact_match` still catches ordering bugs.

In [ ]:
from collections import Counter
from typing import Any

def trajectory_match(outputs, reference_outputs):
    return {
        "key": "exact_match",
        "score": int(outputs["trajectory"] == reference_outputs["trajectory"]),
    }

def extra_steps(outputs, reference_outputs):
    extras = Counter(outputs["trajectory"]) - Counter(reference_outputs["trajectory"])
    return {"key": "extra_steps", "score": sum(extras.values())}

def missing_steps(outputs, reference_outputs):
    missing = Counter(reference_outputs["trajectory"]) - Counter(outputs["trajectory"])
    return {"key": "missing_steps", "score": sum(missing.values())}


Next, we'll define the run function.

In [ ]:
def extract_tool_calls(messages: list[Any]) -> list[str]:
    """Extract tool call names from messages in order."""
    tool_names = []
    for msg in messages:
        if getattr(msg, "tool_calls", None):
            tool_names.extend(tc["name"] for tc in msg.tool_calls)
    return tool_names

def run_agent_trajectory(inputs: dict) -> dict:
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )
    return {"trajectory": extract_tool_calls(result["messages"])}


In [ ]:
results = client.evaluate(
    run_agent_trajectory,
    data=dataset_name,
    evaluators=[trajectory_match, extra_steps, missing_steps],
    experiment_prefix="trajectory",
    max_concurrency=2,
)
print(f"View at: {results.experiment_name}")


## Part 4. Online Evaluations

**Online evals** run automatically against every new trace as it lands in your tracing project — same evaluator as in Part 3, just triggered on incoming runs instead of a dataset.

LangSmith calls these **run rules**. The Python SDK doesn't expose them directly, so we wrap the REST endpoint with a helper at `utils/langsmith_rules.py`. 
Pass in: a project name, an LLM-as-judge prompt, an output schema. Get back: the rule ID and a deep link to inspect it in the UI.

In [ ]:
# Define the LLM-as-judge prompt + schema.
judge_prompt = (
    "You score whether an assistant response satisfied the user's request.\n"
    "Reply with correctness (true/false) and one sentence of comment explaining why."
)

judge_schema = {
    "title": "correctness",
    "description": "Score whether the assistant response was correct/helpful.",
    "type": "object",
    "properties": {
        "correctness": {"type": "boolean", "description": "True if the response was correct/helpful"},
        "comment": {"type": "string", "description": "One short sentence explaining the score"},
    },
    "required": ["correctness", "comment"],
    "strict": True,
}

online_rule = create_run_rule(
    client,
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    display_name="workshop-online-correctness",
    sampling_rate=1.0,
    # Score only root traces, not every child LLM/tool/middleware span.
    filter="eq(is_root, true)",
    llm_judge_prompt=judge_prompt,
    llm_judge_schema=judge_schema,
)

print("Rule ID:", online_rule["id"])
print("Open in UI:", online_rule["url"])


## Annotation Queues — Close the Loop

Once runs have **feedback scores** (from the online eval above, or any other source), route the low-scoring ones to a human for review.

LangSmith's annotation queues are that queue. We use **the same `create_run_rule` helper** — this time with `add_to_annotation_queue_id` set instead of an LLM judge. 
Any run matching the filter is added to the queue automatically.


In [ ]:
queue = get_or_create_annotation_queue(
    client,
    name="modular-workshops-needs-review",
    description="Runs routed here by the workshop's correctness automation rule.",
)
print(f"Queue: {queue.name} (id={queue.id})")


In [ ]:
# Same helper, no evaluator this time -- just a routing rule.
# Filter: only root traces (is_root=true) with correctness > 0.5.
queue_rule = create_run_rule(
    client,
    project_name=os.environ.get("LANGSMITH_PROJECT", "modular-workshops"),
    display_name="workshop-route-correctness",
    sampling_rate=1.0,
    filter=(
        'and('
        'eq(is_root, true), '
        'eq(feedback_key, "correctness"), '
        'gt(feedback_score, 0.5)'
        ')'
    ),
    add_to_annotation_queue_id=queue.id,
)

print("Queue rule ID:", queue_rule["id"])
print("Open in UI:    ", queue_rule["url"])


### Trigger both rules

Both rules are live. Run a few more light traces and you'll see:

1. The online eval fires on each new trace and attaches a `correctness` feedback score (~30s delay).
2. The queue rule fires on each *new feedback* that matches its filter (low correctness) and routes the run to the review queue.


In [ ]:
trigger_prompts = [
    "In one sentence, what is LangSmith? Search at most once.",
    "In one sentence, what does `create_agent` do in LangChain v1? Search at most once.",
    "In one sentence, what is the difference between LangChain and LangGraph? Search at most once.",
]

time.sleep(100)

total = 0.0
for q in trigger_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    total += elapsed
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].content[:120]}")

# Use the tenant_id LangSmith returned with the rule so the link works regardless of workspace.
tenant_id = queue_rule["payload"]["tenant_id"]

print(f"\nTotal: {total:.1f}s.")
print(f"\nOnline eval rule:  {online_rule['url']}")
print(f"Queue rule:        {queue_rule['url']}")
print(f"Queue (review UI): https://smith.langchain.com/o/{tenant_id}/annotation-queues/{queue.id}")
print("\nFeedback shows up in the rule pages within ~30s; queue placements follow once feedback lands.")


### Common run-rule patterns

Swap the `filter` to build different rules:

| Use Case | `filter` |
|---|---|
| Low online correctness | `and(eq(feedback_key, "correctness"), lt(feedback_score, 0.5))` |
| Errored runs | `eq(status, "error")` |
| Slow runs | `gt(latency, 10)` |
| Long-running tool calls | `and(eq(run_type, "tool"), gt(latency, 3))` |
| Specific tool fired | `eq(name, "tavily_search")` |

Both rule types — online eval and queue routing — go through the same `create_run_rule` helper. 
Use `delete_run_rule(client, rule_id)` to tear them down when you're done.


## Recap

| Part | What | API |
|---|---|---|
| **1. Prompt engineering** | Author, version, and share prompts; pull them into code | `client.push_prompt(...)` / `client.pull_prompt(...)` |
| **Warm-up** | Generate a few traces with the Module 2 agent | `agent.invoke(...)` |
| **2. Tracing + querying** | Auto-capture every run; pull back by filter | `LANGSMITH_TRACING=true`, `client.list_runs(filter=...)` |
| **3. Offline evals** | Score on demand against a dataset | `model.with_structured_output(...)` + `client.evaluate` |
| **4. Online evals** | Score every new trace automatically | `create_run_rule(..., llm_judge_prompt=..., llm_judge_schema=...)` |
| **Annotation queues** | Route flagged runs for human review | `create_run_rule(..., add_to_annotation_queue_id=...)` |

The full loop: trace → online eval scores it → run rule routes low scores to the queue → human reviews → fixes flow into the next dataset.

**Next:** Module 5 — **Engine** automates this entire loop (detect → diagnose → PR → evaluator) on your deployed agent.